# Bank Survival Analysis — Customer Churn Timing

## Business Case

Traditional churn classification asks:

> **"Will this customer churn?"**

Survival Analysis asks a richer question:

> **"What is the probability that the customer survives, and when is the churn likely to happen?"**

This notebook builds an end-to-end banking Survival Analysis project using synthetic customer data.

### Objective

Estimate:

1. Customer survival probability over time
2. Churn risk
3. Expected timing of churn
4. Factors associated with churn
5. High-risk customer groups

> Synthetic data is used for education only.

## 1. Survival Analysis Concept

Survival Analysis models the **time until an event occurs**.

```text
Customer joins / observation starts
              ↓
       Customer remains active
              ↓
       ┌──────┴──────┐
       ↓             ↓
    Churn         Still Active
       ↓             ↓
   Event=1       Censored
```

The event can be:

- customer churn,
- loan default,
- account closure,
- first delinquency,
- product cancellation.

### Key difference from classification

Classification:

```text
Churn? → Yes / No
```

Survival Analysis:

```text
Churn?
+
When?
+
Probability of surviving until time t?
```

## 2. Important Terminology

### Event

The event of interest.

Here:

```text
Customer Churn
```

### Duration / Time-to-Event

How long the customer was observed before:

- churn, or
- observation ended.

### Censoring

The event was not observed during the observation window.

Example:

```text
Customer active for 36 months
Observation ends
Customer did not churn
```

We do **not** know whether they would churn later.

This is called **right censoring**.

## 3. Banking Use Cases

### Customer Churn

```text
When will the customer leave?
```

### Loan Default Timing

```text
When might the loan default?
```

### Credit Card Cancellation

```text
How long until card cancellation?
```

### Product Retention

```text
How long will a customer keep a product?
```

### Collection

```text
How long until a delinquency event?
```

Survival analysis is especially useful when **timing matters**, not only whether an event happens.

## 4. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",100)

df=pd.read_csv("bank_customer_survival_sample.csv")

print("Rows:",len(df))
display(df.head())

## 5. Dataset Dictionary

In [ ]:
dictionary=pd.DataFrame({
    "Column":[
        "Customer_ID","Age","Tenure_Months","Monthly_Income",
        "Product_Count","Monthly_Transactions",
        "Digital_Engagement","Complaint_Count",
        "Credit_Utilization","Late_Payment_Count",
        "Segment","Observation_Months","Churn_Event"
    ],
    "Meaning":[
        "Customer identifier","Customer age","Existing tenure",
        "Monthly income","Number of bank products",
        "Monthly transaction volume","Digital engagement score",
        "Complaint count","Credit utilization ratio",
        "Late payment count","Customer segment",
        "Observed survival duration in months",
        "1=churn observed, 0=censored"
    ]
})

display(dictionary)

## 6. Data Quality

In [ ]:
display(df.isna().sum().to_frame("Missing"))
print("Duplicate Customer IDs:",df["Customer_ID"].duplicated().sum())
display(df["Churn_Label"].value_counts())
print("Event rate:",round(df["Churn_Event"].mean(),3))

## 7. Exploratory Analysis

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
sns.histplot(
    data=df,
    x="Observation_Months",
    hue="Churn_Label",
    bins=30,
    element="step",
    stat="density",
    common_norm=False,
    ax=ax
)
ax.set_title("Observed Time-to-Event Distribution")
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
sns.boxplot(data=df,x="Churn_Label",y="Digital_Engagement",ax=ax)
ax.set_title("Digital Engagement by Churn Status")
plt.show()

## 8. Why Censoring Matters

Suppose we observe:

```text
Customer A → churned after 18 months
Customer B → active after 36 months
```

For Customer A:

```text
duration = 18
event = 1
```

For Customer B:

```text
duration = 36
event = 0
```

Customer B is not necessarily a "non-churner".

We simply did not observe the churn event during the available period.

This is why ordinary classification can lose important information.

## 9. Kaplan-Meier Estimator

In [ ]:
kmf=KaplanMeierFitter()

kmf.fit(
    durations=df["Observation_Months"],
    event_observed=df["Churn_Event"],
    label="All Customers"
)

fig,ax=plt.subplots(figsize=(10,6))
kmf.plot_survival_function(ax=ax)
ax.set_title("Kaplan-Meier Customer Survival Curve")
ax.set_xlabel("Months")
ax.set_ylabel("Probability Customer Remains Active")
plt.show()

## 10. Interpreting the Survival Curve

The y-axis represents:

> Probability that a customer has **not yet churned** by time t.

Example:

```text
S(12) = 0.85
```

would mean approximately 85% of customers are estimated to remain active through month 12.

The exact values should be read from the fitted dataset/model, not assumed in advance.

In [ ]:
for month in [6,12,24,36,48]:
    try:
        print(
            f"Survival at {month} months:",
            round(float(kmf.predict(month)),3)
        )
    except:
        pass

## 11. Median Survival Time

The median survival time is the time at which estimated survival reaches 50%.

If the curve never reaches 50%, the median survival time is not estimable from the observed data.

In [ ]:
print("Median survival time:",kmf.median_survival_time_)

## 12. Survival by Customer Segment

A major benefit of Survival Analysis is comparing groups.

Question:

> Do customer segments have different retention patterns?

We use separate Kaplan-Meier curves.

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))

for segment in df["Segment"].unique():
    subset=df[df["Segment"]==segment]
    fitter=KaplanMeierFitter()
    fitter.fit(
        subset["Observation_Months"],
        subset["Churn_Event"],
        label=segment
    )
    fitter.plot_survival_function(ax=ax)

ax.set_title("Customer Survival by Segment")
ax.set_xlabel("Months")
ax.set_ylabel("Survival Probability")
plt.show()

## 13. Log-Rank Test

The log-rank test compares survival curves between groups.

For example:

```text
H0:
Survival distributions are the same.
```

A small p-value indicates evidence that the survival curves differ.

Statistical significance does not automatically mean the difference is operationally important.

In [ ]:
mass=df[df["Segment"]=="Mass"]
affluent=df[df["Segment"]=="Affluent"]

test=logrank_test(
    mass["Observation_Months"],
    affluent["Observation_Months"],
    event_observed_A=mass["Churn_Event"],
    event_observed_B=affluent["Churn_Event"]
)

print("Test statistic:",round(test.test_statistic,3))
print("p-value:",test.p_value)

## 14. Cox Proportional Hazards Model

The Cox model estimates how features affect **hazard**.

Conceptually:

```text
Hazard
=
instantaneous risk of event
given that the customer has survived until now
```

The model estimates a **hazard ratio (HR)**.

Interpretation:

```text
HR > 1 → higher hazard
HR < 1 → lower hazard
HR = 1 → no estimated association
```

This is an association model, not proof of causality.

## 15. Prepare Features for Cox Model

We use:

- Age
- Tenure
- Income
- Product count
- Transactions
- Digital engagement
- Complaints
- Credit utilization
- Late payments
- Segment

Customer_ID is excluded because it has no predictive meaning.

In [ ]:
cox_df=df.drop(columns=["Customer_ID","Churn_Label"]).copy()

cox_df=pd.get_dummies(
    cox_df,
    columns=["Segment"],
    drop_first=True
)

cox_df=cox_df.astype(float)

display(cox_df.head())

## 16. Train Cox Model

In [ ]:
cph=CoxPHFitter()

cph.fit(
    cox_df,
    duration_col="Observation_Months",
    event_col="Churn_Event"
)

cph.print_summary()

## 17. Hazard Ratio Interpretation

Suppose a feature has:

```text
Hazard Ratio = 1.30
```

The model estimates a higher hazard associated with a one-unit increase in that feature, holding other included variables constant.

If:

```text
Hazard Ratio = 0.70
```

the estimated hazard is lower.

For continuous variables, the unit scale matters. Therefore, business interpretation should consider realistic changes rather than blindly interpreting a one-unit change.

In [ ]:
hr=cph.summary[[
    "exp(coef)","exp(coef) lower 95%","exp(coef) upper 95%","p"
]].sort_values("exp(coef)",ascending=False)

hr.columns=["Hazard_Ratio","HR_Lower_95","HR_Upper_95","P_Value"]

display(hr.round(3))

## 18. Visualize Hazard Ratios

In [ ]:
fig,ax=plt.subplots(figsize=(9,7))
cph.plot(hazard_ratios=True,ax=ax)
ax.set_title("Cox Model Hazard Ratios")
plt.show()

## 19. Proportional Hazards Assumption

The Cox model assumes that the hazard ratio between groups is approximately constant over time.

This assumption should be checked before production use.

Violation of the assumption can require:

- time-varying covariates,
- stratification,
- alternative survival models.

In [ ]:
# Diagnostic check
# lifelines provides a built-in diagnostic.
# It may print warnings for variables that violate the assumption.
cph.check_assumptions(
    cox_df,
    p_value_threshold=0.05,
    show_plots=False
)

## 20. Individual Survival Prediction

Once fitted, the Cox model can estimate a survival curve for an individual customer.

This answers:

> "Given this customer's characteristics, how does their estimated probability of remaining active change over time?" 

In [ ]:
customer=cox_df.iloc[[0]].drop(columns=["Churn_Event","Observation_Months"])

survival_curve=cph.predict_survival_function(customer)

fig,ax=plt.subplots(figsize=(10,6))
survival_curve.plot(ax=ax)
ax.set_title("Example Customer Survival Prediction")
ax.set_xlabel("Months")
ax.set_ylabel("Predicted Survival Probability")
plt.show()

## 21. Predicted Churn Risk at a Specific Horizon

Instead of predicting only:

```text
Churn = Yes / No
```

we can ask:

```text
Probability of surviving 12 months?
Probability of surviving 24 months?
```

Therefore:

```text
Churn Risk at 12 months
= 1 - Survival Probability at 12 months
```

In [ ]:
predicted_survival=cph.predict_survival_function(
    cox_df.drop(columns=["Churn_Event","Observation_Months"])
)

horizons=[6,12,24,36]

risk_table=pd.DataFrame(index=range(len(df)))

for h in horizons:
    survival_at_h=predicted_survival.loc[
        predicted_survival.index.get_indexer([h],method="nearest")[0]
    ]
    risk_table[f"Churn_Risk_{h}M"]=1-survival_at_h.values

display(risk_table.head().round(3))

## 22. Customer Risk Segmentation

A survival model can support operational segmentation:

```text
Low estimated churn risk
        ↓
Normal engagement

Medium risk
        ↓
Targeted retention

High risk
        ↓
Retention intervention
```

The threshold should be selected using business costs, capacity, validation data, and policy—not arbitrarily.

In [ ]:
risk_12=risk_table["Churn_Risk_12M"]

risk_segment=pd.cut(
    risk_12,
    bins=[-np.inf,.25,.50,np.inf],
    labels=["Low","Medium","High"]
)

risk_summary=pd.DataFrame({
    "Customer_ID":df["Customer_ID"],
    "Risk_12M":risk_12,
    "Risk_Band":risk_segment
})

display(risk_summary.head(20))
display(risk_summary["Risk_Band"].value_counts().to_frame("Customers"))

## 23. Survival Analysis vs Classification

| Classification | Survival Analysis |
|---|---|
| Churn yes/no | Churn + timing |
| Often ignores censoring | Explicitly handles censoring |
| Fixed prediction horizon | Multiple horizons |
| Probability of class | Survival / hazard estimates |
| "Will churn?" | "When might churn occur?" |

Both can be useful.

The correct choice depends on the business question.

## 24. Survival Analysis vs Forecasting

Forecasting:

```text
What will aggregate demand be next month?
```

Survival Analysis:

```text
When will this customer churn?
```

Forecasting generally models a time series or aggregate future values.

Survival Analysis models **time-to-event at the individual or cohort level**.

## 25. Business Use Case — Retention

A possible workflow:

```text
Customer Base
      ↓
Survival Model
      ↓
12M Churn Risk
      +
Expected Timing
      ↓
Customer Prioritization
      ↓
Retention Campaign
      ↓
Customer Response
      ↓
Measure Outcome
```

The model identifies risk; the retention strategy should be evaluated separately.

## 26. Alternative Use Case — Loan Default Timing

The same methodology can be applied to loans.

Replace:

```text
Customer Churn
```

with:

```text
Loan Default
```

Then:

```text
Duration = Months Since Loan Origination
Event = Default
Censoring = Loan Still Performing at Observation End
```

Potential features:

- DPD history
- utilization
- income
- outstanding balance
- payment ratio
- loan age
- customer tenure.

## 27. Model Evaluation

For Survival Analysis, conventional accuracy is usually not enough.

Common metrics include:

### Concordance Index (C-index)

Measures whether higher predicted risk tends to correspond to earlier events.

### Brier Score

Measures prediction error over time.

### Calibration

Checks whether predicted survival/risk aligns with observed outcomes.

### Time-dependent AUC

Evaluates discrimination at specific time horizons.

In [ ]:
print("Cox concordance index:",round(cph.concordance_index_,3))

## 28. Common Mistakes

1. Treating censored customers as non-churners.
2. Ignoring observation duration.
3. Using future information as a feature.
4. Confusing hazard with probability.
5. Interpreting correlation as causation.
6. Ignoring proportional hazards assumptions.
7. Evaluating only one time horizon.
8. Using arbitrary risk thresholds.
9. Ignoring calibration.
10. Deploying without monitoring.

## 29. Production Architecture

```text
Customer / Loan Data
        ↓
Feature Engineering
        ↓
Survival Model
        ↓
Risk at 3 / 6 / 12 / 24 Months
        ↓
Customer / Loan Prioritization
        ↓
CRM / Collection / Retention
        ↓
Outcome Tracking
        ↓
Model Monitoring
        ↓
Retraining
```

For banking production systems, add:

- model governance,
- audit trails,
- data quality checks,
- access control,
- explainability,
- monitoring,
- champion/challenger validation.

## 30. Final Executive Summary

### Business Question

> **When is the customer likely to churn?**

### Solution

```text
Customer Data
     ↓
Survival Analysis
     ↓
Kaplan-Meier
     ↓
Cox Proportional Hazards
     ↓
Survival Probability
     +
Churn Risk
     +
Timing
```

### Key Advantage

Instead of only saying:

```text
Customer A → Churn = Yes
```

we can estimate:

```text
Customer A
→ estimated survival probability over time
→ estimated churn risk at specific horizons
→ relative risk factors
```

### Banking Applications

- Customer churn timing
- Loan default timing
- Credit card cancellation
- Product retention
- Delinquency timing
- Collection prioritization

> **Survival Analysis is especially useful when the timing of an event is as important as whether the event happens.**